# Read distance from the 3′ end of single-isoform genes

For every read that maps to a gene with exactly one annotated isoform we
calculate how far the read start is from the gene's 3′ end, measured **only
over exonic bases** (intronic regions are excluded from the count).

## Strand-aware distance formula

Because `pysam` always reports the leftmost (`reference_start`, 0-based) and
one-past-rightmost (`reference_end`, 0-based exclusive / 1-based inclusive)
genomic coordinates regardless of strand, we convert to 1-based coordinates
and choose the appropriate read anchor per strand:

| Strand | 3′ end of gene | Read anchor (1-based) | Distance formula |
|--------|----------------|-----------------------|------------------|
| `+`    | `gene_end`     | `read.reference_start + 1` | `(gene_end − read_pos + 1) − intronic_bases(read_pos, gene_end)` |
| `−`    | `gene_start`   | `read.reference_end`       | `(read_pos − gene_start + 1) − intronic_bases(gene_start, read_pos)` |

All genomic coordinates are 1-based and inclusive (GTF convention).

In [ ]:
import re
import pysam
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict

## 1. Paths — adjust as needed

In [ ]:
# Output of the Snakemake rule `filter_single_isoform_genes`
single_isoform_genes_path = (
    "/tmp/Mazutislab-out/Ignas/RT_comparison/genome_and_annotations/single_isoform_genes.txt"
)

# GTF annotation file used by the Snakemake rule
gtf_path = (
    "/tmp/Mazutislab-out/Ignas/RT_comparison/genome_and_annotations/"
    "gencode.v41.primary_assembly.annotation.gtf.filtered"
)

# One BAM file per sample (output of the star_mapping rule)
out_dir = "/tmp/Mazutislab-out/Ignas/RT_comparison"
samples = [
    "25_SSCV_KG_01_S1",
    "25_SSCV_KG_02_S2",
    "25_SSCV_KG_03_S3",
    "25_SSCV_KG_04_S4",
]
bam_paths = {
    s: f"{out_dir}/{s}/star/{s}_Aligned.sortedByCoord.out.bam" for s in samples
}

## 2. Load single-isoform gene list

In [ ]:
# Maps gene_id -> transcript_id for all genes with exactly one isoform.
# Also builds gene_id -> gene_name for convenience.
gene_id_to_transcript = {}
gene_id_to_name = {}

with open(single_isoform_genes_path) as fh:
    for line in fh:
        parts = line.strip().split(" ")
        gene_id, gene_name, transcript_id = parts[0], parts[1], parts[2]
        gene_id_to_transcript[gene_id] = transcript_id
        gene_id_to_name[gene_id] = gene_name

print(f"Loaded {len(gene_id_to_transcript):,} single-isoform genes.")

## 3. Build per-gene coordinate dictionary from the GTF

For each single-isoform gene we store:
- `strand` — `'+'` or `'-'`
- `gene_start`, `gene_end` — gene-level 1-based coordinates from the GTF `gene` feature
- `exon_coords` — sorted list of `(start, end)` 1-based tuples for the single transcript's exons
- `intron_coords` — derived from consecutive exon pairs

In [ ]:
# Initialise entries for every gene we care about.
gene_info = {
    gene_id: {
        "transcript_id": transcript_id,
        "strand": None,
        "gene_start": None,
        "gene_end": None,
        "exon_coords": [],
        "intron_coords": [],
    }
    for gene_id, transcript_id in gene_id_to_transcript.items()
}

with open(gtf_path) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue

        feature_type = fields[2]
        if feature_type not in ("gene", "exon"):
            continue

        attr = fields[8]
        m = re.search(r'gene_id "([^"]+)"', attr)
        if not m:
            continue
        gene_id = m.group(1)

        if gene_id not in gene_info:
            continue

        start = int(fields[3])   # 1-based inclusive
        end = int(fields[4])     # 1-based inclusive
        strand = fields[6]

        if feature_type == "gene":
            gene_info[gene_id]["strand"] = strand
            gene_info[gene_id]["gene_start"] = start
            gene_info[gene_id]["gene_end"] = end

        elif feature_type == "exon":
            # Only collect exons belonging to the single isoform of this gene.
            m_t = re.search(r'transcript_id "([^"]+)"', attr)
            if m_t and m_t.group(1) == gene_info[gene_id]["transcript_id"]:
                gene_info[gene_id]["exon_coords"].append((start, end))

# Sort exons by start coordinate and derive intron intervals.
for gene_id, info in gene_info.items():
    exons = sorted(info["exon_coords"], key=lambda x: x[0])
    info["exon_coords"] = exons
    introns = []
    for i in range(1, len(exons)):
        intron_start = exons[i - 1][1] + 1
        intron_end = exons[i][0] - 1
        if intron_end >= intron_start:   # sanity check
            introns.append((intron_start, intron_end))
    info["intron_coords"] = introns

# Report how many genes were fully annotated (have gene-level coordinates).
n_complete = sum(1 for info in gene_info.values() if info["gene_start"] is not None)
print(f"Gene info built for {n_complete:,} / {len(gene_info):,} single-isoform genes.")

### Quick sanity check on a single gene

In [ ]:
example_gene = next(iter(gene_info))
print(f"Gene: {example_gene}  ({gene_id_to_name[example_gene]})")
info = gene_info[example_gene]
print(f"  strand     : {info['strand']}")
print(f"  gene_start : {info['gene_start']}")
print(f"  gene_end   : {info['gene_end']}")
print(f"  exon_coords: {info['exon_coords']}")
print(f"  intron_coords: {info['intron_coords']}")

## 4. Helper functions

In [ ]:
def intronic_bases_between(intron_coords, pos1, pos2):
    """
    Count intronic bases that fall within the closed genomic interval [pos1, pos2]
    (1-based, inclusive). Both pos1 and pos2 are assumed to be within the gene.
    """
    count = 0
    for intron_start, intron_end in intron_coords:
        overlap_start = max(intron_start, pos1)
        overlap_end = min(intron_end, pos2)
        if overlap_end >= overlap_start:
            count += overlap_end - overlap_start + 1
    return count


def distance_from_3prime(read, info):
    """
    Compute the exonic distance (in bases) from the read's transcript-start
    position to the gene's 3′ end, excluding intronic bases.

    Parameters
    ----------
    read : pysam.AlignedSegment
    info : dict   — entry from gene_info for the gene the read maps to

    Returns
    -------
    int or None
        Distance in exonic bases, or None if the read anchor falls outside
        the gene boundaries.
    """
    strand = info["strand"]
    gene_start = info["gene_start"]
    gene_end = info["gene_end"]
    intron_coords = info["intron_coords"]

    if strand == "+":
        # 3′ end is at gene_end (highest coordinate).
        # Read anchor: leftmost mapped base, converted to 1-based.
        read_pos = read.reference_start + 1
        if read_pos < gene_start or read_pos > gene_end:
            return None
        n_intronic = intronic_bases_between(intron_coords, read_pos, gene_end)
        return (gene_end - read_pos + 1) - n_intronic

    else:  # strand == "-"
        # 3′ end is at gene_start (lowest coordinate).
        # Read anchor: rightmost mapped base.
        # pysam reference_end is 0-based exclusive, which equals the 1-based
        # inclusive end of the last aligned base.
        read_pos = read.reference_end
        if read_pos < gene_start or read_pos > gene_end:
            return None
        n_intronic = intronic_bases_between(intron_coords, gene_start, read_pos)
        return (read_pos - gene_start + 1) - n_intronic

## 5. Iterate over BAM files and compute distances

In [ ]:
# For each sample, collect the exonic distances from the 3′ end.
distances_per_sample = {}

for sample, bam_path in bam_paths.items():
    print(f"\nProcessing {sample} ...")
    bam = pysam.AlignmentFile(bam_path, "rb")

    distances = []
    n_total = 0
    n_no_gene_tag = 0
    n_not_single_isoform = 0
    n_out_of_bounds = 0

    for read in tqdm.tqdm(bam.fetch()):
        # Skip unmapped, secondary, supplementary reads.
        if read.is_unmapped or read.is_secondary or read.is_supplementary:
            continue

        n_total += 1

        # Get the gene ID assigned by STARsolo (GX tag).
        try:
            gene_id = read.get_tag("GX")
        except KeyError:
            n_no_gene_tag += 1
            continue

        # Skip reads not assigned to a single-isoform gene.
        if gene_id not in gene_info:
            n_not_single_isoform += 1
            continue

        dist = distance_from_3prime(read, gene_info[gene_id])
        if dist is None:
            n_out_of_bounds += 1
            continue

        distances.append(dist)

    bam.close()
    distances_per_sample[sample] = np.array(distances)

    print(f"  total primary reads         : {n_total:,}")
    print(f"  missing GX tag              : {n_no_gene_tag:,}")
    print(f"  not in single-isoform genes : {n_not_single_isoform:,}")
    print(f"  read out of gene bounds     : {n_out_of_bounds:,}")
    print(f"  reads used in statistic     : {len(distances):,}")

## 6. Plot the distribution of distances from the 3′ end

In [ ]:
fig, axes = plt.subplots(len(samples), 1, figsize=(10, 3 * len(samples)), sharex=True)

max_dist = max(
    (d.max() for d in distances_per_sample.values() if len(d) > 0),
    default=5000,
)
bin_edges = np.arange(0, min(max_dist + 100, 10001), 100)

for ax, sample in zip(axes, samples):
    dists = distances_per_sample.get(sample, np.array([]))
    ax.hist(dists, bins=bin_edges, color="steelblue", edgecolor="none")
    ax.set_title(sample)
    ax.set_ylabel("Number of reads")
    ax.set_xlim(0, bin_edges[-1])

axes[-1].set_xlabel("Exonic distance from 3′ end (bases)")
fig.suptitle(
    "Distribution of read distances from the 3′ end\n"
    "(single-isoform genes, introns excluded)",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics per sample
print(f"{'Sample':<30} {'N reads':>10} {'Median dist':>12} {'Mean dist':>10}")
print("-" * 65)
for sample in samples:
    dists = distances_per_sample.get(sample, np.array([]))
    if len(dists) == 0:
        print(f"{sample:<30} {'0':>10} {'NA':>12} {'NA':>10}")
    else:
        print(
            f"{sample:<30} {len(dists):>10,} "
            f"{np.median(dists):>12.1f} {np.mean(dists):>10.1f}"
        )